# Agentic Healthcare Assistant for Medical Task Automation
**Applied Generative AI Specialisation – Capstone**

A virtual medical assistant that **books appointments**, **manages medical records**, **summarises medical histories**
and **searches trusted medical sources** (MedlinePlus, WHO). It is built as a LangGraph agent with a planner,
tool executor, FAISS-backed long-term memory and a RAG pipeline, plus an LLMOps layer (evaluation, logging) and
a Streamlit dashboard.

This notebook walks through the system in the order of the problem statement:

| Part | Step | Section |
|---|---|---|
| 1 | Agent planning & goal decomposition | §4 |
| 1 | Tool & memory setup (Doctor Schedule API, EHR DB, web search, FAISS, memory) | §2–3 |
| 1 | Prompt engineering & task chaining | §4 |
| 1 | Agent execution flow (sample scenario) | §5 |
| 2 | Model evaluation (QAEvalChain + metrics) | §6 |
| 2 | Streamlit dashboard, memory & logs interface | §7 |

> Runs without an API key (deterministic offline mode). Put `GROQ_API_KEY` (or `OPENAI_API_KEY` / `ANTHROPIC_API_KEY`)
> in `.env` to switch the planner, summariser, RAG answerer and evaluator to a real LLM.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_colwidth", 120)

from healthcare_agent.config import get_settings
from healthcare_agent.llm import get_llm
print("LLM:", get_llm().label)
print("Dataset:", sorted(p.name for p in get_settings().raw_dir.iterdir()))

LLM: offline (heuristic)
Dataset: ['course_requirements.txt', 'records.xlsx', 'sample_patient.pdf', 'sample_report_anjali.pdf', 'sample_report_david.pdf', 'sample_report_ramesh.pdf']


## 1. Dataset & ingestion
The dataset has `records.xlsx` (one row **per encounter** – Rebeca Nagle appears three times) and two PDF layouts:
simple *History & Physical* notes, and a 12-page EHR export for Rebeca Nagle containing progress notes, a
past-medical-history page and CCD documents with **lab results + reference ranges** and **vital signs**.
Custom parsers turn both into structured rows (encounters, ICD-10 diagnoses, medications, flagged labs, vitals).

In [2]:
raw = pd.read_excel(get_settings().raw_dir / "records.xlsx")
print(raw.shape, "-> unique patients:", raw.drop_duplicates(["Name", "Phone_number"]).shape[0])
raw[["Name", "Age", "Gender", "Address", "Summary"]]

(7, 7) -> unique patients: 5


,Name,Age,Gender,Address,Summary
0,Rahul Negi,31,Male,Chattarpur,Rahul is a fit and healthy person. He is doing well in his life
1,Rebeca Nagle,36,Female,"9125 XYZ Hill St, Tigard,OR97223",NaN
2,Rebeca Nagle,36,Female,"9125 XYZ Hill St, Tigard,OR97223",NaN
3,Rebeca Nagle,36,Female,"9125 XYZ Hill St, Tigard,OR97223",NaN
4,Ramesh Kulkarni,65,Male,"52 Residency Road, Chennai",Patient presents for routine checkup with a history of hypertension. Vitals are stable. Continue current medication ...
5,Anjali Mehra,33,Female,"202 Lakeview Apartments, Pune","Patient presents with 5-day history of dry cough and mild fever, diagnosed with Upper Respiratory Infection (J06.9),..."
6,David Thompson,51,Male,"17 MG Road, Indiranagar, Bangalore",Patient presents for follow-up of Type 2 Diabetes. Reports increased thirst and urination. Diagnosis: Type 2 Diabete...


In [3]:
from healthcare_agent.ingest import parse_pdf
doc = parse_pdf(get_settings().raw_dir / "sample_patient.pdf")
print("layout:", doc.layout, "| patient:", doc.patient)
for e in doc.encounters:
    print(f"  encounter {e.encounter_id} {e.visit_date}: {e.chief_complaint} -> {[d['description'] + ' ' + d['icd10'] for d in e.diagnoses]}")
labs = pd.DataFrame(doc.labs)
print(f"{len(labs)} lab results parsed; out of range:")
labs[labs.flag.notna()][["taken_on", "test", "value", "unit", "ref_range", "flag"]]

layout: ehr_export | patient: {'name': 'Rebeca Nagle', 'dob': '1986-05-19', 'gender': 'Female', 'mrn': 'D2147783321-907568', 'address': '9125 XYZ Hill St, Tigard,OR97223', 'phone': '+1-541-950-0000'}
  encounter 727374050 2022-09-07: wellness, PAP -> ['Well adult exam Z00.00', 'Cervical cancer screening Z12.4']
  encounter 730177633 2022-09-20: Body Ache / Pain-zocdoc-MW -> ['Costochondritis M94.0']
45 lab results parsed; out of range:


,taken_on,test,value,unit,ref_range,flag
0,2022-09-08,AST,34.0,U/L,10-30,H
12,2022-09-08,ALT,52.0,U/L,6-29,H
15,2022-09-08,Glucose,104.0,mg/dL,65-99,H
39,2022-09-08,Triglycerides,190.0,mg/dL,<150,H
42,2022-09-08,HDL cholesterol,35.0,mg/dL,> OR = 50,L


### Build the EHR database and vector indexes
`build()` loads the spreadsheet, parses every PDF, links documents to patients, adds the synthetic
**doctor directory + rolling slot calendar** (the "Doctor Schedule API" backend – not part of the dataset) and one
synthetic demo patient needed by the reference scenario (**Suresh Negi, 70, CKD stage 3b – father of Rahul Negi**;
flagged `source=synthetic_demo`), then embeds everything into FAISS.

In [4]:
from healthcare_agent.build import build
stats = build(reset=True)
stats

records.xlsx -> 5 unique patients


sample_patient.pdf [ehr_export] -> P002: 2 encounters, 45 labs
sample_report_anjali.pdf [hp_note] -> P004: 1 encounters, 0 labs
sample_report_david.pdf [hp_note] -> P005: 1 encounters, 0 labs


sample_report_ramesh.pdf [hp_note] -> P003: 1 encounters, 0 labs


synthetic demo patient Suresh Negi -> P006 (father of Rahul Negi)
doctors: 15, slots created: 2010


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3832.44it/s]

patient_memory index: 16 chunks


medical_knowledge index: 262 chunks from seed corpus


{'patients': 6, 'slots': 2010, 'patient_chunks': 16, 'medical_chunks': 262}

In [5]:
from healthcare_agent import db, records
pd.DataFrame(records.list_patients())[["patient_id", "name", "age", "gender", "source", "allergies"]]

,patient_id,name,age,gender,source,allergies
0,P001,Rahul Negi,31,Male,records.xlsx,NaN
1,P002,Rebeca Nagle,40,Female,records.xlsx,NaN
2,P003,Ramesh Kulkarni,67,Male,records.xlsx,NaN
3,P004,Anjali Mehra,36,Female,records.xlsx,NaN
4,P005,David Thompson,54,Male,records.xlsx,NaN
5,P006,Suresh Negi,70,Male,synthetic_demo,sulfa drugs (rash).


In [6]:
pd.DataFrame(db.query('''SELECT d.doctor_id, d.name, d.specialty, d.location, d.work_start || '-' || d.work_end AS hours,
       SUM(s.status='available') AS open_slots, SUM(s.status='blocked') AS held_slots
FROM doctors d JOIN slots s USING(doctor_id) GROUP BY d.doctor_id'''))

,doctor_id,name,specialty,location,hours,open_slots,held_slots
0,D01,Dr. Priya Raman,Nephrology,"Sunrise Multispeciality Clinic, Chennai",09:00-13:00,77,43
1,D02,Dr. Vikram Sethi,Nephrology,"Lakeside Health Centre, Bangalore",14:00-18:00,65,31
2,D03,Dr. Ananya Iyer,Cardiology,"Sunrise Multispeciality Clinic, Chennai",10:00-16:00,94,50
3,D04,Dr. Rohan Kapoor,Endocrinology,"Lakeside Health Centre, Bangalore",09:00-15:00,103,41
4,D05,Dr. Meera Nair,General Medicine,"CityCare Family Clinic, Pune",08:30-17:00,207,99
5,D06,Dr. Sameer Khan,General Medicine,"Lakeside Health Centre, Bangalore",09:00-17:00,166,74
6,D07,Dr. Kavya Reddy,Pulmonology,"CityCare Family Clinic, Pune",10:00-14:00,54,18
7,D08,Dr. Elena Brooks,Obstetrics & Gynecology,"Bridport Family Medicine, Portland",08:00-12:00,88,32
8,D09,Dr. Arvind Menon,Neurology,"Sunrise Multispeciality Clinic, Chennai",13:00-17:00,48,24
9,D10,Dr. Farah Siddiqui,Rheumatology,"Lakeside Health Centre, Bangalore",09:00-13:00,41,31


## 2. Tools (the agent's APIs)
Every tool is a plain Python function wrapped by `traced_tool`, which logs arguments, result, success and latency
to SQLite (the source of the tool-success metrics). Business failures (no slot, not authorised) return
`{"ok": False, "error": ...}` so the executor can react.

* **Doctor Schedule API** – `find_doctors`, `check_availability`, `book_appointment` (atomic, idempotent),
  `cancel_appointment`, `reschedule_appointment` (book-then-release), `list_appointments`
* **EHR / Patient DB** – `identify_patient` (relation / fuzzy name / self), `get_patient_history`,
  `search_patient_notes`, staff-only `register_patient`, `update_patient_record`, `add_clinical_note`
* **Medical search** – `medical_info_search`: MedlinePlus Web Service + WHO (site-restricted search & fact-sheet
  extraction) → FAISS → grounded, cited LLM answer

Access control: caregivers see only themselves and linked dependents; only attendants/doctors can edit records.

In [7]:
from healthcare_agent.tools import catalog_text
print(catalog_text())

- identify_patient(name?, relation? (father/mother/spouse/son/daughter/self), age?): Resolve which patient the request is about (by relation to the user, by name, or self).
- get_patient_history(focus? (e.g. 'kidney disease')): Retrieve and summarise the patient's diagnoses, treatments, medications and alerts from the EHR.
- search_patient_notes(query): Semantic search over the patient's past notes and remembered facts.
- find_doctors(specialty?, condition?, location?): List doctors for a specialty or condition.
- check_availability(specialty?, condition?, doctor_id?, preferred_date?, time_of_day?): List open appointment slots for a specialty/doctor and optional date/time preference.
- book_appointment(specialty?, condition?, doctor_id?, preferred_date?, time_of_day?, reason?): Book the earliest matching slot for the patient (idempotent; will not double-book).
- cancel_appointment(appointment_id?, specialty?): Cancel an appointment by id, or the patient's next appointment (optionally o

In [8]:
from healthcare_agent.context import Actor, current_actor
from healthcare_agent.tools.patient import identify_patient, get_patient_history
current_actor.set(Actor("P001", "Rahul Negi", "patient"))
print(identify_patient(relation="father"))
print(identify_patient(name="Ramesh Kulkarni"))   # not his dependent -> privacy guard

{'ok': True, 'patient': {'patient_id': 'P006', 'name': 'Suresh Negi', 'age': 70, 'gender': 'Male', 'phone': '+91-98110-45671', 'address': 'Chattarpur, New Delhi'}, 'method': 'relation:father'}
{'ok': False, 'error': "Rahul Negi is not authorised to access this patient's records (not self or a linked dependent)."}


In [9]:
from healthcare_agent.tools.scheduling import check_availability
res = check_availability(specialty="Nephrology", preferred_date="next week", time_of_day="morning", limit=4)
pd.DataFrame(res["slots"])[["doctor_name", "start_ts", "location"]]

,doctor_name,start_ts,location
0,Dr. Priya Raman,2026-09-28 09:00:00,"Sunrise Multispeciality Clinic, Chennai"
1,Dr. Priya Raman,2026-09-28 10:30:00,"Sunrise Multispeciality Clinic, Chennai"
2,Dr. Priya Raman,2026-09-28 11:00:00,"Sunrise Multispeciality Clinic, Chennai"
3,Dr. Priya Raman,2026-09-28 11:30:00,"Sunrise Multispeciality Clinic, Chennai"


## 3. Memory
| Memory | Where | Used for |
|---|---|---|
| Short-term (conversation) | LangGraph state persisted per `thread_id` by the checkpointer | follow-ups: *"also book a dietitian **for him**"* |
| Long-term (patient) | SQLite `memories` + FAISS `patient_memory` (profiles, encounter chunks, learned facts) | context recall before planning, history summaries |
| Knowledge | FAISS `medical_knowledge` (MedlinePlus / WHO chunks, cached) | RAG answers |

Embeddings: `BAAI/bge-small-en-v1.5` (normalised → cosine similarity via `IndexFlatIP`).

In [10]:
from healthcare_agent import memory
for h in memory.recall("falling kidney function and high potassium", k=3, min_score=0):
    print(round(h["score"], 3), h["metadata"]["patient_name"], "|", h["text"][:110])

0.742 Suresh Negi | [Suresh Negi] Visit 2026-08-13 (CityCare Family Clinic, Pune). Complaint: Follow-up of kidney function; fatigu
0.71 Suresh Negi | [Suresh Negi] Visit 2026-03-01 (CityCare Family Clinic, Pune). Complaint: Routine diabetes and blood pressure 
0.678 Suresh Negi | Patient: Suresh Negi (ID P006), 70y Male, DOB 1956-03-02
Allergies: sulfa drugs (rash).
Diagnoses: Chronic kid


## 4. Planning, goal decomposition & prompt chaining
The graph: `safety → context → planner → executor (one tool per visit, loops) → synthesizer → memory`.

* **Planner** – structured-output LLM call returning a typed `Plan` (sub-goals, tool, args, dependencies).
  The system prompt injects the tool catalogue, today's date, the acting user, their dependents, the active patient
  and **recalled memories** (memory lookup in the prompt). Offline, a keyword planner produces the same schema.
* **Plan validation/repair** (applies to any plan) – drops unknown/duplicate tools, removes staff-only steps for
  patients, inserts a missing `identify_patient` step, wires dependencies.
* **Chain**: planner → tools → history-summary prompt / medical-RAG prompt → synthesis prompt → memory-extraction prompt.

In [11]:
from healthcare_agent import prompts
print(prompts.PLANNER_SYSTEM[:1800], "...")

You are part of a healthcare *administration* assistant. You do not diagnose, prescribe or replace a clinician. Never invent patient data, appointment times, doctors or sources; use only what the tools returned. If information is missing, say so.

ROLE: You are the PLANNER. Decompose the user's request into an ordered list of sub-goals, each
fulfilled by exactly one tool call. You never answer the user yourself.

AVAILABLE TOOLS:
{tool_catalog}

PLANNING RULES:
1. Patient-specific work (history, booking, records, appointments) must start with `identify_patient`.
   Fill `name` if the user names the patient, `relation` if they refer to a relative ("my father"),
   or relation="self" when they talk about themselves. Omit identify_patient for pure information questions.
2. Retrieve history (`get_patient_history`) before booking when the booking is about a medical condition,
   so the appointment reason and the doctor briefing carry clinical context.
3. For booking, set `specialty` when th

In [12]:
from healthcare_agent.agent.planner import heuristic_plan, llm_plan, validate_plan
SAMPLE = ("My 70-year-old father has chronic kidney disease. I want to book a nephrologist for him. "
          "Also, can you summarize latest treatment methods?")
actor = Actor("P001", "Rahul Negi", "patient")
ctx = {"dependents": [{k: d[k] for k in ("relation", "patient_id", "name", "age")} for d in records.get_dependents("P001")],
       "active_patient": None, "memories": [], "conversation": []}
plan, mode = llm_plan(SAMPLE, actor, ctx) if get_llm().live else (heuristic_plan(SAMPLE, actor, ctx), "offline")
plan, repairs = validate_plan(plan, actor, ctx)
print("planner mode:", mode, "| repairs:", repairs)
pd.DataFrame([{"step": s.step, "sub-goal": s.goal, "tool": s.tool,
               "args": s.args.model_dump(exclude_none=True), "depends_on": s.depends_on} for s in plan.steps])

planner mode: offline | repairs: []


,step,sub-goal,tool,args,depends_on
0,1,Identify the patient (father),identify_patient,"{'relation': 'father', 'age': 70}",[]
1,2,Retrieve and summarise the patient's medical history,get_patient_history,{'focus': 'chronic kidney disease'},[1]
2,3,Book a Nephrology appointment,book_appointment,"{'specialty': 'Nephrology', 'condition': 'chronic kidney disease', 'reason': 'chronic kidney disease consultation'}",[1]
3,4,Search trusted sources and summarise the medical information,medical_info_search,{'query': 'chronic kidney disease latest treatment'},[]


## 5. Agent execution – the reference scenario
Expected workflow from the problem statement: **(1)** identify patient & context → **(2)** retrieve father's
history → **(3)** query doctor calendar & book → **(4)** search & summarise treatments via RAG.

In [13]:
from healthcare_agent.agent import HealthcareAgent
agent = HealthcareAgent()
result = agent.run(SAMPLE, actor, thread_id="notebook-demo")
print(f"success={result['success']}  steps={result['steps_ok']}/{result['steps_total']}  latency={result['latency_ms']} ms")
pd.DataFrame([{"step": s["step"], "tool": s["tool"], "status": s["status"], "ms": s["ms"]} for s in result["steps"]])

success=True  steps=4/4  latency=7479 ms


,step,tool,status,ms
0,1,identify_patient,success,14
1,2,get_patient_history,success,90
2,3,book_appointment,success,54
3,4,medical_info_search,success,7074


In [14]:
from IPython.display import Markdown
Markdown(result["answer"])

**Patient:** Suresh Negi (P006), 70 y, Male

### Medical history - Suresh Negi
**Overview**: Suresh Negi, 70-year-old male. Active problems: Chronic kidney disease stage 3b, Hyperkalemia, Type 2 diabetes mellitus with diabetic chronic kidney disease, Hypertensive chronic kidney disease.
**Diagnoses & treatments**:
- 2026-08-13: Follow-up of kidney function; fatigue - Start Empagliflozin 10mg OD. Continue Losartan 50mg OD; recheck potassium in 2 weeks. Reduce Metformin to 500mg OD given eGFR. Refer to nephrology. Dietitian review for low-potassium diet.
- 2026-03-01: Routine diabetes and blood pressure review - Start Losartan 50mg OD. Continue Metformin 500mg BID. Low-salt diet. Repeat renal function in 6 months.
**Current medications**: Empagliflozin 10mg OD, Metformin 500mg OD, Losartan 50mg OD, Atorvastatin 20mg HS.
**Alerts**: Allergy: sulfa drugs (rash); BUN HIGH: 32 mg/dL (ref 7-25, 2026-08-13); Creatinine HIGH: 1.9 mg/dL (ref 0.70-1.30, 2026-08-13); HbA1c HIGH: 7.4 % (ref <5.7, 2026-08-13); Hemoglobin LOW: 11.2 g/dL (ref 13.5-17.5, 2026-08-13); Potassium HIGH: 5.4 mmol/L (ref 3.5-5.3, 2026-08-13); Urine albumin/creatinine ratio HIGH: 180 mg/g (ref <30, 2026-08-13); eGFR LOW: 36 mL/min/1.73m2 (ref > OR = 60, 2026-08-13); Geriatric patient (age 70): review dosing and fall risk.

### Appointment booked
- **Doctor:** Dr. Priya Raman (Nephrology)
- **When:** Mon 28 Sep 2026, 09:00 AM
- **Where:** Sunrise Multispeciality Clinic, Chennai
- **Appointment ID:** `APT-B2B0E292`

### Medical information: chronic kidney disease latest treatment
Key points from trusted sources (extractive summary):
- Chronic kidney disease damages the nephrons slowly over several years. [3]
- The management of CKD focuses on slowing progression, reducing cardiovascular risk, and preventing complications of advanced kidney disease. [2]
- For treatment of kidney failure, kidney replacement therapy (dialysis or kidney transplantation) may be required. [2]
- Integrating kidney health into primary care is essential for early diagnosis and timely treatment. [2]
- In advanced or severe kidney disease, symptoms may include fatigue, breathlessness, generalized itchiness, leg swelling, muscle cramps, nausea or vomiting. [2]
- NIH: National Institute of Diabetes and Digestive and Kidney Diseases [3]
- Treatments cannot cure kidney disease , but they may slow kidney disease . [1]

Discuss any treatment changes with the treating doctor.

**Sources:**
1. [Chronic Kidney Disease](https://medlineplus.gov/chronickidneydisease.html) - MedlinePlus (NIH/NLM)
2. [Kidney disease - World Health Organization (WHO)](https://www.who.int/news-room/fact-sheets/detail/kidney-disease) - World Health Organization
3. [Kidney Diseases](https://medlineplus.gov/kidneydiseases.html) - MedlinePlus (NIH/NLM)
4. [Diabetic Kidney Problems](https://medlineplus.gov/diabetickidneyproblems.html) - MedlinePlus (NIH/NLM)

_This assistant supports scheduling and information access; it does not replace advice from your doctor._

In [15]:
# Graph path with per-node timings (the same trace the Streamlit "Memory & logs" page shows)
pd.DataFrame([{"node": t["node"], "ms": t["ms"], "detail": json.dumps(t["detail"], default=str)[:150]} for t in result["trace"]])

,node,ms,detail
0,safety,1,"{""level"": ""none"", ""label"": null, ""message"": """"}"
1,context,55,"{""dependents"": 1, ""memories_recalled"": 4, ""active_patient"": null, ""prior_turns"": 0}"
2,planner,0,"{""mode"": ""offline"", ""steps"": [""1. identify_patient"", ""2. get_patient_history"", ""3. book_appointment"", ""4. medical_in..."
3,executor,14,"{""step"": 1, ""tool"": ""identify_patient"", ""status"": ""success"", ""reason"": null}"
4,executor,90,"{""step"": 2, ""tool"": ""get_patient_history"", ""status"": ""success"", ""reason"": null}"
5,executor,54,"{""step"": 3, ""tool"": ""book_appointment"", ""status"": ""success"", ""reason"": null}"
6,executor,7074,"{""step"": 4, ""tool"": ""medical_info_search"", ""status"": ""success"", ""reason"": null}"
7,synthesizer,7,"{""mode"": ""offline"", ""chars"": 3010}"
8,memory,152,"{""written"": [""2026-09-27: Rahul Negi (patient) asked: \""My 70-year-old father has chronic kidney disease. I want to ..."


### Multi-turn follow-up (short-term memory) and long-term memory written

In [16]:
follow = agent.run("Also book a dietitian for him next week in the morning", actor, thread_id="notebook-demo")
print(follow["repairs"])
print(follow["steps"][-1]["result"]["appointment"])
pd.DataFrame(memory.list_memories(result["active_patient"]["patient_id"]))[["created_at", "kind", "content"]]

['using active patient from conversation: Suresh Negi']
{'appointment_id': 'APT-F822FE9B', 'patient_id': 'P006', 'doctor_id': 'D15', 'doctor_name': 'Dr. Aditya Rao', 'specialty': 'Nutrition & Dietetics', 'location': 'Lakeside Health Centre, Bangalore', 'start_ts': '2026-09-28 10:30:00', 'reason': 'Nutrition & Dietetics consultation', 'status': 'booked'}


,created_at,kind,content
0,2026-09-27 22:20:36,interaction,"2026-09-27: Rahul Negi (patient) asked: ""Also book a dietitian for him next week in the morning"". Outcome: reviewed ..."
1,2026-09-27 22:20:35,interaction,"2026-09-27: Rahul Negi (patient) asked: ""My 70-year-old father has chronic kidney disease. I want to book a nephrolo..."
2,2026-09-27 22:20:35,fact,Reported by Rahul Negi: My 70-year-old father has chronic kidney disease.


### Guard-rails

In [17]:
for who, q in [(Actor("P001", "Rahul Negi", "patient"), "My father has chest pain and is sweating and short of breath"),
              (Actor("P004", "Anjali Mehra", "patient"), "Update my address to 14 MG Road, Pune"),
              (Actor("P004", "Anjali Mehra", "patient"), "Show me the medical history of Ramesh Kulkarni")]:
    r = agent.run(q, who, thread_id="guard-" + who.user_id + q[:5])
    print(f"> {q}\n  safety={r['safety']['level']} repairs={r['repairs']}\n  {r['answer'][:230]!r}\n")

> My father has chest pain and is sweating and short of breath
  safety=emergency repairs=['emergency: deferred medical information search']
  '> **This may be an emergency (possible heart attack).** Please call your local emergency number now (India 112 / US 911) or go to the nearest emergency department. Do not wait for a scheduled appointment.\n\n\n_This assistant support'



> Update my address to 14 MG Road, Pune
  safety=none repairs=['removed update_patient_record: requires attendant/doctor role']
  '**Patient:** Anjali Mehra (P004), 36 y, Female\n\n**Need more information:** Updating medical records requires an attendant or doctor (you are signed in as patient).\n\n_This assistant supports scheduling and information access; it do'



> Show me the medical history of Ramesh Kulkarni
  safety=none repairs=[]
  "**Could not complete: Identify the patient (Ramesh Kulkarni)** - Anjali Mehra is not authorised to access this patient's records (not self or a linked dependent).\n\n**Retrieve and summarise the patient's medical history** - skipped"



## 6. Evaluation (LLMOps)
`run_evaluation()` scores each module against golden sets and stores case-level rows in `eval_results`:

* **planner** – tool precision/recall/F1, exact match, specialty accuracy, patient-identification accuracy (15 queries)
* **summary** – generated summaries vs the `Summary` column of records.xlsx: semantic similarity, ROUGE-L recall,
  token F1, and **QAEvalChain** CORRECT/INCORRECT when an LLM is configured
* **medical_qa** – RAG answers vs references: similarity, key-fact coverage, groundedness (share of answer
  sentences supported by retrieved passages), citation presence, **QAEvalChain**
* **retrieval** – FAISS hit@1/@3 and MRR for patient memory; hit@3 for the knowledge base
* **booking** – success vs expected outcome, constraint satisfaction, idempotency, race safety (bookings rolled back)
* **tools** – live per-tool success rate & latency from the logs

In [18]:
from healthcare_agent.evaluation.runner import run_evaluation
report = run_evaluation(verbose=False)
rows = [{"module": m, "metric": k, "mean": v} for m, mets in report["summary"].items() for k, v in mets.items()]
pd.DataFrame(rows)

,module,metric,mean
0,planner,tool_precision,1.0000
1,planner,tool_recall,1.0000
2,planner,tool_f1,1.0000
3,planner,exact_match,1.0000
4,planner,specialty_accuracy,1.0000
5,planner,patient_id_accuracy,1.0000
6,summary,semantic_similarity,0.8649
7,summary,rouge_l_recall,0.7511
8,summary,token_f1,0.3606
9,medical_qa,answered,1.0000


In [19]:
import plotly.graph_objects as go
df = pd.DataFrame(rows); df = df[df.module != "tools"]
fig = go.Figure(go.Bar(y=df.module + " · " + df.metric, x=df["mean"], orientation="h", marker_color="#2a78d6",
                       text=df["mean"].round(2), textposition="outside"))
fig.update_layout(title=f"Evaluation – {report['llm']}", height=620, xaxis_range=[0, 1.15],
                  yaxis_autorange="reversed", margin=dict(l=10, r=10, t=40, b=10), plot_bgcolor="white")
fig

In [20]:
cases = pd.DataFrame(report["rows"])
cases[cases.module == "medical_qa"].pivot_table(index="case_id", columns="metric", values="score").round(2)

metric,answered,cited_sources,context_recall,groundedness,keyword_coverage,semantic_similarity
case_id,,,,,,
Q01,1.0,1.0,1.0,0.88,1.00,0.88
Q02,1.0,1.0,1.0,0.88,1.00,0.85
Q03,1.0,1.0,1.0,0.88,0.25,0.80
Q04,1.0,1.0,1.0,0.88,0.75,0.77
Q05,1.0,1.0,1.0,0.88,0.00,0.79
Q06,1.0,1.0,1.0,0.88,0.33,0.79
Q07,1.0,1.0,1.0,0.88,1.00,0.92
Q08,1.0,1.0,1.0,0.88,1.00,0.70


## 7. Streamlit dashboard
```bash
streamlit run app.py
```
| Page | Requirement covered |
|---|---|
| 💬 Chat with the assistant | live plan + execution trace per turn, example prompts per role |
| 🧪 Scenario tester | interactive elements to test user scenarios (11 curated, incl. guard-rails) |
| 🧑 Patient view | record, alerts, lab trends, appointments, memory; staff: update fields, add notes, upload PDFs, register |
| 🩺 Doctor view | calendar utilisation, upcoming patients with clinical briefing |
| 📅 Appointment tracker | real-time (auto-refresh) tracking, filters, cancellation |
| 📚 Medical information | trusted-source RAG search, latest retrieved information, knowledge base |
| 📊 Evaluation & metrics | tool success & latency, booking success, LLM call modes, evaluation suite runs |
| 🧠 Memory & logs | agent memory traces, planning breakdowns, tool / LLM logs with success-failure filters |